# Train the Task 1 BERT classifier on a Colab GPU

Use this when your own machine cannot fine-tune BERT (for example an 8 GB Mac).
It trains exactly what notebook 06 trains (`src/classify/bert.py`, frozen split, seed 42).

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** Colab asks once for access to your Google Drive: allow it.
3. Wait until the **last cell prints `DONE`** (about 15 minutes). Keep this tab open.
4. The model is saved as **`bert_intent.zip` in the top level of your Google Drive (My Drive)**. The browser may also download it directly.
5. Download it from drive.google.com and give it to whoever runs notebook 06 (unzip into `outputs/bert_intent/`).

If a cell shows a red error, stop and send a screenshot of it.

In [ ]:
# Step 1/5: GPU check
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all again"
print("Step 1/5 OK:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# Step 2/5: connect Google Drive (the model is saved there so it survives a disconnect)
from google.colab import drive
drive.mount("/content/drive")
print("Step 2/5 OK: Drive connected")

In [ ]:
# Step 3/5: get the code and the frozen split
BRANCH = "dev/zfang75"
!rm -rf /content/banking-support-agent
!git clone -q -b {BRANCH} https://github.com/sh1or1tama1/banking-support-agent.git /content/banking-support-agent
%cd /content/banking-support-agent
!pip -q install "accelerate>=1.1" "datasets>=2.18"
!git log --oneline -1
print("Step 3/5 OK: code ready")

In [ ]:
# Step 4/5: fine-tune BERT (about 10-15 minutes on a T4; progress bar below)
import sys, time
sys.path.insert(0, "/content/banking-support-agent")
import pandas as pd, transformers
from src import config as C
from src.utils import set_seed
from src.classify.bert import train_bert
set_seed()
df = pd.read_parquet(C.SPLITS_FILE)
train, val = df[df["split"] == "train"], df[df["split"] == "val"]
print(len(train), "train |", len(val), "val | transformers", transformers.__version__)
t0 = time.time()
model, tok, labels, history = train_bert(train, val)
history.to_csv(C.OUTPUTS / "bert_intent" / "training_history.csv", index=False)
print(f"Step 4/5 OK: training took {(time.time() - t0) / 60:.1f} min")
history[["epoch", "eval_accuracy", "eval_macro_f1", "eval_loss"]]

In [ ]:
# Step 5/5: zip the final model (not the checkpoint-* folders), save it to Google Drive, and try a direct download
import os, shutil, zipfile
src = C.OUTPUTS / "bert_intent"
keep = [f for f in sorted(os.listdir(src)) if (src / f).is_file()]
print("files:", keep)
with zipfile.ZipFile("/content/bert_intent.zip", "w", zipfile.ZIP_STORED) as z:
    for f in keep:
        z.write(src / f, arcname=f)
shutil.copy("/content/bert_intent.zip", "/content/drive/MyDrive/bert_intent.zip")
print(f"saved to Google Drive: My Drive/bert_intent.zip ({os.path.getsize('/content/bert_intent.zip') / 1e6:.0f} MB)")
try:
    from google.colab import files
    files.download("/content/bert_intent.zip")
except Exception as e:
    print("direct download did not start; use Google Drive instead:", e)
print("DONE")